# m6A cleaning with fixed train/test outputs
## Goal
Process dataset0, dataset1 and dataset2 **separately**, preserving the original cleaning rules and 45 signal summaries. Use `split_manifest.csv` as the source of binary labels and split assignments. Original `train` + `validation` become `train`; original `test` stays `test`. No random split is generated.

Each dataset produces two main files: `<dataset>_clean_train.csv.gz` and `<dataset>_clean_test.csv.gz`, plus small QC/audit files and a feature allowlist. CSV.gz is compressed CSV; `pandas.read_csv()` reads it directly.

The manifest's `label` is used unchanged. `modification_fraction` is retained as metadata, not converted again. Original `data.info` files are not required or independently reconciled by this version.

## Setup
Use Python with NumPy, pandas, Matplotlib and ipykernel. Edit the four input paths and output folder in section 1, then **Restart Kernel → Run All**. Absolute Windows or Ronin paths both work. Raw JSON files are required; the manifest alone does not contain signal features.

### Manifest inspected during preparation
213,971 manifest rows; no duplicate `(dataset, transcript_id, transcript_position)` keys. Before cleaning, train/test counts are dataset0: 97,879/23,959; dataset1: 73,380/17,430; dataset2: 1,015/308. These are input counts, not observed cleaned output counts.

No supplied group IDs or nonmissing gene IDs cross the resulting train/test boundary. Seven transcript IDs do cross it across the supplied manifest; inspect the generated transcript audit before claiming transcript-disjoint evaluation. Dataset2 uses `assumed_shared_position` groups and anonymised transcript IDs: these are **not verified biological gene groups**. Missing genes remain missing.

This version accepts **three raw JSON files, three original label files, and one shared split CSV**. Label files validate the agreed manifest targets; they never silently replace them. Set paths in section 1, then Run All. Outputs are written to a new timestamped folder.

In [ ]:
 # Uncomment if needed, then restart the kernel.
%pip install numpy pandas matplotlib ipykernel

In [ ]:

from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import json, gzip, re, sys, platform, importlib.metadata, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print  # Allows execution in a plain Python environment too.
plt.rcParams.update({'figure.figsize': (10, 4), 'font.size': 11})
pd.set_option('display.max_columns', 12)

import hashlib, warnings


## 1. Input paths and settings
Change these example paths to your actual locations. Keep all three entries to generate six main CSVs. `MAX_LINES=None` processes complete files; a numeric limit is only a smoke test and will leave many manifest rows absent.

`EXPORT_CLEAN_READS=True` additionally exports separate train/test JSON files for models requiring individual reads. The default CSV outputs contain site summaries, exactly as in the earlier notebook.

Each label file needs `transcript_id`, `transcript_position`, and `label` (CSV, including comma-separated `.info`; TSV also supported). `label_type="auto"` uses fraction comparison when the manifest has modification fractions for that dataset, otherwise binary comparison. Override with `binary` or `fraction` if needed. A fractional source is checked against `modification_fraction`; it is never re-thresholded. If a dataset has no separate label file, explicitly set `labels_path=None` to rely on the manifest. Unknown or conflicting labels stop the run with an audit.

In [ ]:
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'dataset/raw/split_manifest.csv').is_file())
SPLIT_MANIFEST = PROJECT_ROOT / 'dataset/raw' / 'split_manifest.csv'
OUTPUT_ROOT = PROJECT_ROOT / 'dataset/rebuilt/gru'
DATASETS = [
    {'dataset_id': 'dataset0', 'json_path': PROJECT_ROOT / 'dataset/raw/data0/dataset0_json/dataset0.json',
     'labels_path': PROJECT_ROOT / 'dataset/raw/data0/data.info.labelled',
     'label_type': 'auto'},
    {'dataset_id': 'dataset1', 'json_path': PROJECT_ROOT / 'dataset/raw/data1/dataset1_json/dataset1.json',
     'labels_path': PROJECT_ROOT / 'dataset/raw/data1/data.info',
     'label_type': 'auto'},
    {'dataset_id': 'dataset2', 'json_path': PROJECT_ROOT / 'dataset/raw/data2/dataset2_json/dataset2.json',
     'labels_path': PROJECT_ROOT / 'dataset/raw/data2/data.info',
     'label_type': 'auto'},
]
MAX_LINES = None
MIN_VALID_READS = 1
REQUIRE_DRACH = True
EXPORT_CLEAN_READS = False
KEY = ['transcript_id', 'transcript_position']
JOIN_KEY = ['dataset_id'] + KEY
SIGNALS = [f'{location}_{measure}' for location in ('minus1', 'center', 'plus1')
           for measure in ('dwell', 'signal_sd', 'signal_mean')]
STATS = ('mean', 'std', 'median', 'q25', 'q75')
FEATURES = [f'{name}_{stat}' for stat in STATS for name in SIGNALS]
MODEL_FEATURES = FEATURES + ['n_reads_valid', 'central_5mer']


## 2. Validate the manifest before reading large JSON files
Reject missing identifiers, invalid positions or binary labels, unknown splits and all duplicate site keys (including exact duplicates). Preserve metadata and `original_split`. Check supplied groups and known genes globally across datasets; an overlap stops the run rather than silently changing your assignments. Transcript overlaps are saved and warned about because dataset2 identifiers need not represent real transcripts.

One row remains one **transcript position**, not one transcript. `n_reads` is the original manifest count; `n_reads_raw` and `n_reads_valid` describe the JSON before and after read cleaning.

In [ ]:
def resolve_input(value):
    p = Path(value).expanduser()
    return (p if p.is_absolute() else PROJECT_ROOT / p).resolve()

def load_split_manifest(path):
    m = pd.read_csv(path, dtype=str, keep_default_na=False)
    required = {'dataset', 'transcript_id', 'transcript_position', 'label',
                'gene_id', 'n_reads', 'group_id', 'group_source',
                'modification_fraction', 'split'}
    if not required.issubset(m.columns):
        raise ValueError(f'Missing columns: {sorted(required - set(m.columns))}')
    # Select the defined schema so unexpected metadata cannot collide with features.
    m = m[list(sorted(required))].rename(columns={'dataset': 'dataset_id', 'split': 'original_split'})
    for col in ['dataset_id', 'transcript_id', 'group_id', 'group_source']:
        if m[col].str.strip().eq('').any():
            raise ValueError(f'Empty {col}')
        if m[col].ne(m[col].str.strip()).any():
            raise ValueError(f'Whitespace around {col}; correct source explicitly')
    for col in ['transcript_position', 'n_reads']:
        if not m[col].str.fullmatch(r'\d+').all():
            raise ValueError(f'{col} must be nonnegative integers')
        m[col] = m[col].astype('int64')
    label = pd.to_numeric(m.label, errors='coerce')
    if not label.isin([0, 1]).all():
        raise ValueError('Manifest label must be binary 0/1 with no missing values')
    m['label'] = label.astype('int8')
    f = pd.to_numeric(m.modification_fraction.replace('', np.nan), errors='coerce')
    if (m.modification_fraction.ne('') & (f.isna() | ~f.between(0, 1))).any():
        raise ValueError('Invalid modification_fraction')
    m['modification_fraction'] = f
    m['gene_id'] = m.gene_id.replace(r'^\s*$', pd.NA, regex=True)
    if not m.original_split.isin(['train', 'validation', 'test']).all():
        raise ValueError('Expected split values: train, validation, test')
    if m.duplicated(JOIN_KEY, keep=False).any():
        raise ValueError('Duplicate manifest site keys: resolve before processing')
    m['split'] = m.original_split.replace({'validation': 'train'})
    if m.empty:
        raise ValueError('Empty manifest')
    return m

def crossing_ids(m, column):
    counts = m.dropna(subset=[column]).groupby(column)['split'].nunique()
    return counts.index[counts.gt(1)]

ids = [d['dataset_id'] for d in DATASETS]
if not ids or len(ids) != len(set(ids)) or any(not re.fullmatch(r'[A-Za-z0-9_-]+', x) for x in ids):
    raise ValueError('Provide unique, filename-safe dataset IDs')
if MIN_VALID_READS < 1 or (MAX_LINES is not None and MAX_LINES < 1):
    raise ValueError('Invalid read/line limit')
manifest_path = resolve_input(SPLIT_MANIFEST)
for p in [manifest_path] + [resolve_input(d['json_path']) for d in DATASETS] + [resolve_input(d['labels_path']) for d in DATASETS if d.get('labels_path') is not None]:
    if not p.is_file():
        raise FileNotFoundError(f'Edit section 1: {p}')
manifest = load_split_manifest(manifest_path)
for name in ids:
    if name not in set(manifest.dataset_id):
        raise ValueError(f'{name} absent from split manifest')
for col in ['group_id', 'gene_id']:
    overlap = crossing_ids(manifest, col)
    if len(overlap):
        raise ValueError(f'{len(overlap)} {col} values cross train/test; review manifest')
transcript_overlaps = manifest.loc[manifest.transcript_id.isin(crossing_ids(manifest, 'transcript_id'))]
if len(transcript_overlaps):
    warnings.warn('Transcript IDs cross train/test. Review transcript_overlap_audit.csv; supplied groups are preserved.')
RUN_ROOT = OUTPUT_ROOT / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
RUN_ROOT.mkdir(parents=True, exist_ok=False)
transcript_overlaps.to_csv(RUN_ROOT / 'transcript_overlap_audit.csv', index=False)
manifest_counts = manifest.groupby(['dataset_id', 'original_split', 'split'], observed=True).agg(
    sites=('label', 'size'), positives=('label', 'sum'), groups=('group_id', 'nunique')).reset_index()
manifest_counts.to_csv(RUN_ROOT / 'manifest_split_counts.csv', index=False)
MANIFEST_SHA256 = hashlib.sha256(manifest_path.read_bytes()).hexdigest()
display(manifest_counts)
print('Output folder:', RUN_ROOT)


## 3. Read validation and feature extraction
Each read must have nine finite numeric values. Dwell times and signal SDs must be nonnegative. Identical vectors are counted but retained because distinct reads may share values. Extreme finite values are retained.

Across valid reads at each site, calculate 45 summaries: mean, population SD (`ddof=0`), median, 25th and 75th percentiles for each measurement. These are within-site summaries, not dataset-wide standardisation. They lose read-level structure; use the optional cleaned JSON when a model requires individual reads.

Malformed JSON, repeated JSON keys or an unexpected schema stop with a line number. Repeated site keys are quarantined in every occurrence, rather than merged.


In [ ]:
def open_text(path):
    return gzip.open(path, 'rt', encoding='utf-8') if path.suffix == '.gz' else path.open(encoding='utf-8')

def unique_object(pairs):
    result = {}
    for k, v in pairs:
        if k in result:
            raise ValueError(f'Duplicate JSON key: {k!r}')
        result[k] = v
    return result

def read_matrix(reads):
    if not isinstance(reads, list):
        raise ValueError('Expected a list of reads')
    valid, reasons = [], Counter()
    for read in reads:
        if not isinstance(read, list) or len(read) != 9:
            reasons['wrong_length'] += 1; continue
        if any(isinstance(v, bool) or not isinstance(v, (int, float)) for v in read):
            reasons['nonnumeric'] += 1; continue
        a = np.asarray(read, dtype=float)
        if not np.isfinite(a).all():
            reasons['nonfinite'] += 1; continue
        if (a[[0, 1, 3, 4, 6, 7]] < 0).any():
            reasons['negative_dwell_or_sd'] += 1; continue
        valid.append(a)
    return np.asarray(valid, dtype=float).reshape(-1, 9), reasons

def parse_sites(path, max_lines=None):
    records, totals = [], Counter()
    with open_text(path) as handle:
        for line_no, line in enumerate(handle, 1):
            if max_lines is not None and line_no > max_lines: break
            totals['lines_scanned'] += 1
            if not line.strip():
                totals['blank_lines'] += 1; continue
            try:
                obj = json.loads(line, object_pairs_hook=unique_object)
                if not isinstance(obj, dict) or len(obj) != 1:
                    raise ValueError('Expected exactly one transcript per line')
                transcript, positions = next(iter(obj.items()))
                if not transcript.strip() or not isinstance(positions, dict) or len(positions) != 1:
                    raise ValueError('Expected one position and a nonempty transcript ID')
                pos, sequences = next(iter(positions.items()))
                if not re.fullmatch(r'\d+', pos):
                    raise ValueError('Position must be a nonnegative integer string')
                if not isinstance(sequences, dict) or len(sequences) != 1:
                    raise ValueError('Expected exactly one sequence per site')
                sequence, reads = next(iter(sequences.items()))
                a, reasons = read_matrix(reads)
                row = {'transcript_id': transcript, 'transcript_position': int(pos),
                       'sequence': sequence, 'central_5mer': sequence[1:6],
                       'source_line': line_no, 'n_reads_raw': len(reads),
                       'n_reads_valid': len(a), 'n_reads_removed': len(reads)-len(a),
                       'valid_sequence': bool(re.fullmatch('[ACGT]{7}', sequence)),
                       'is_drach': bool(re.fullmatch('[AGT][AG]AC[ACT]', sequence[1:6])),
                       'n_identical_vectors': len(a)-len(np.unique(a, axis=0))}
                for reason in ('wrong_length','nonnumeric','nonfinite','negative_dwell_or_sd'):
                    row['removed_'+reason] = reasons[reason]
                values = ([a.mean(0), a.std(0), np.median(a, axis=0),
                           np.quantile(a, .25, axis=0), np.quantile(a, .75, axis=0)]
                          if len(a) else [np.full(9, np.nan)]*5)
                row.update({f'{name}_{stat}': float(v)
                            for stat, arr in zip(STATS, values)
                            for name, v in zip(SIGNALS, arr)})
                records.append(row)
            except (ValueError, TypeError, OverflowError) as exc:
                raise ValueError(f'Cannot parse line {line_no}: {exc}') from exc
            if line_no % 10000 == 0: print(f'Parsed {line_no:,} lines')
    if not records: raise ValueError('No site records found')
    return pd.DataFrame(records), dict(totals)


## 4. Clean, attach assignments, and export each dataset
Keep the original QC rules: invalid reads are removed, repeated site keys are quarantined in every occurrence, and invalid sequences/non-DRACH sites/too few valid reads/nonfinite summaries are excluded. Identical read vectors and extreme finite measurements remain retained.

Join on **dataset + transcript ID + position**, never row order. Fixed per-site cleaning does not learn anything from the test population. Sites absent from the manifest go into an audit file, never an invented split. Manifest sites absent from JSON are distinguished from sites removed by QC. Neither issue is silently filled with zeros. Read-count differences are reported without automatic exclusion.

Each dataset's folder has exactly two main cleaned CSVs. Other CSVs are audit tables, not additional training sets. A partition with no rows or one class is clearly marked in the summary; inspect it before training/evaluation.

In [ ]:
def safe_version(package):
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return 'not available'

def export_clean_reads(source, destination, retained_keys):
    with open_text(source) as src, gzip.open(destination, 'wt', encoding='utf-8') as dst:
        for line_no, line in enumerate(src, 1):
            if MAX_LINES is not None and line_no > MAX_LINES:
                break
            if not line.strip():
                continue
            obj = json.loads(line, object_pairs_hook=unique_object)
            transcript, positions = next(iter(obj.items()))
            position, sequences = next(iter(positions.items()))
            if (transcript, int(position)) in retained_keys:
                sequence, reads = next(iter(sequences.items()))
                valid, _ = read_matrix(reads)
                dst.write(json.dumps({transcript: {position: {sequence: valid.tolist()}}}) + '\n')



In [ ]:
def validate_source_labels(config, selected, run_dir):
    """Validate source labels against the manifest; do not learn or change targets."""
    label_path = config.get('labels_path')
    if label_path is None:
        warnings.warn(f"{config['dataset_id']}: no source label file; using manifest labels")
        return {'mode': 'manifest_only'}
    path = resolve_input(label_path)
    raw = pd.read_csv(path, sep=None, engine='python', dtype=str, keep_default_na=False)
    required = set(KEY + ['label'])
    if not required.issubset(raw):
        raise ValueError(f'{path}: labels require {sorted(required)}')
    dataset_col = 'dataset_id' if 'dataset_id' in raw else 'dataset' if 'dataset' in raw else None
    if dataset_col:
        raw = raw.loc[raw[dataset_col].eq(config['dataset_id'])].copy()
    if raw.empty:
        raise ValueError(f'{path}: no labels for {config["dataset_id"]}')
    position = pd.to_numeric(raw.transcript_position, errors='coerce')
    value = pd.to_numeric(raw.label, errors='coerce')
    valid = (raw.transcript_id.str.strip().ne('') & raw.transcript_id.eq(raw.transcript_id.str.strip())
             & position.notna() & position.ge(0) & position.lt(2**63)
             & position.mod(1).eq(0) & value.between(0, 1))
    raw.loc[~valid].to_csv(run_dir / 'invalid_source_labels.csv', index=False)
    if not valid.all():
        raise ValueError(f'{path}: invalid labels; inspect invalid_source_labels.csv')
    labels = raw[KEY].copy()
    labels['transcript_position'] = position.astype('int64')
    labels['source_label'] = value.astype(float)
    before = len(labels)
    labels = labels.drop_duplicates()
    conflicts = labels.duplicated(KEY, keep=False)
    labels.loc[conflicts].to_csv(run_dir / 'conflicting_source_labels.csv', index=False)
    if conflicts.any():
        raise ValueError(f'{path}: conflicting source labels; inspect audit')
    mode = config.get('label_type', 'auto')
    if mode == 'auto':
        mode = 'fraction' if selected.modification_fraction.notna().any() else 'binary'
    if mode not in ('binary', 'fraction'):
        raise ValueError('label_type must be auto, binary, or fraction')
    if mode == 'binary' and not labels.source_label.isin([0, 1]).all():
        raise ValueError(f'{path}: fractional labels cannot be treated as binary')
    target = 'label' if mode == 'binary' else 'modification_fraction'
    audit = selected.merge(labels, on=KEY, how='outer', validate='one_to_one', indicator=True)
    audit['label_check'] = np.where(audit['_merge'].eq('left_only'), 'missing_source_label',
        np.where(audit['_merge'].eq('right_only'), 'source_only',
        np.where(audit[target].isna(), 'missing_manifest_reference',
        np.where(np.isclose(audit.source_label, audit[target], rtol=0, atol=1e-8), 'match', 'mismatch'))))
    audit.to_csv(run_dir / 'source_label_audit.csv.gz', index=False)
    failures = audit.label_check.isin(['missing_source_label', 'missing_manifest_reference', 'mismatch'])
    if failures.any():
        raise ValueError(f'{path}: {int(failures.sum())} manifest/source label problems; inspect source_label_audit.csv.gz')
    return {'path': str(path), 'sha256': hashlib.sha256(path.read_bytes()).hexdigest(),
            'comparison': mode, 'equivalent_duplicates_collapsed': before-len(labels),
            'checks': {str(k): int(v) for k,v in audit.label_check.value_counts().items()}}


In [ ]:
def process_dataset(config):
    name = config['dataset_id']
    source = resolve_input(config['json_path'])
    run_dir = RUN_ROOT / name
    run_dir.mkdir(exist_ok=False)
    print(f'Processing {name}')
    selected = manifest.loc[manifest.dataset_id.eq(name)].copy()
    source_label_checks = validate_source_labels(config, selected, run_dir)
    sites, parse_counts = parse_sites(source, MAX_LINES)
    sites.insert(0, 'dataset_id', name)
    flags = pd.DataFrame({
        'duplicate_site': sites.duplicated(JOIN_KEY, keep=False),
        'invalid_sequence': ~sites.valid_sequence,
        'non_drach': (~sites.is_drach) & REQUIRE_DRACH,
        'too_few_reads': sites.n_reads_valid < MIN_VALID_READS,
        'nonfinite_summary': ~np.isfinite(sites[FEATURES]).all(axis=1)})
    sites['qc_reason'] = flags.apply(lambda r: ';'.join(r.index[r]), axis=1)
    selected = manifest.loc[manifest.dataset_id.eq(name)].copy()
    audited = sites.merge(selected, on=JOIN_KEY, how='left', validate='many_to_one', indicator=True)
    clean = audited.loc[audited.qc_reason.eq('') & audited['_merge'].eq('both')].drop(columns='_merge').copy()
    clean['label'] = clean.label.astype('int8')
    clean['n_reads'] = clean.n_reads.astype('int64')
    assert not clean.duplicated(JOIN_KEY).any()
    assert np.isfinite(clean[FEATURES]).all().all()
    assert not len(crossing_ids(clean, 'group_id'))
    assert not len(crossing_ids(clean, 'gene_id'))
    # One audit row per manifest key, even if the raw JSON repeats a key.
    status = sites[JOIN_KEY + ['qc_reason']].drop_duplicates(JOIN_KEY)
    coverage = selected.merge(status, on=JOIN_KEY, how='left', validate='one_to_one', indicator=True)
    coverage['status'] = np.where(coverage['_merge'].eq('left_only'), 'missing_from_json',
                                  np.where(coverage.qc_reason.eq(''), 'retained', 'removed_by_qc'))
    coverage = coverage.drop(columns='_merge')
    coverage.to_csv(run_dir / 'manifest_site_audit.csv.gz', index=False)
    audited.loc[audited.qc_reason.ne('')].drop(columns='_merge').to_csv(
        run_dir / 'quarantined_sites.csv.gz', index=False)
    audited.loc[audited['_merge'].eq('left_only'), JOIN_KEY + ['qc_reason']].to_csv(
        run_dir / 'raw_sites_without_manifest.csv.gz', index=False)
    mismatch = audited['_merge'].eq('both') & audited.n_reads.ne(audited.n_reads_raw)
    audited.loc[mismatch, JOIN_KEY + ['n_reads', 'n_reads_raw', 'n_reads_valid']].to_csv(
        run_dir / 'read_count_mismatches.csv', index=False)
    counts = coverage.groupby(['original_split', 'split', 'status'], observed=True).size().rename('sites').reset_index()
    counts.to_csv(run_dir / 'cleaning_counts_by_split.csv', index=False)
    flags.sum().rename('sites').to_csv(run_dir / 'qc_reason_counts.csv')
    (run_dir / 'feature_columns.json').write_text(json.dumps(MODEL_FEATURES, indent=2))
    summaries = []
    for split in ['train', 'test']:
        part = clean.loc[clean.split.eq(split)].sort_values(KEY).reset_index(drop=True)
        path = run_dir / f'{name}_clean_{split}.csv.gz'
        part.to_csv(path, index=False)
        if EXPORT_CLEAN_READS:
            export_clean_reads(source, run_dir / f'{name}_clean_{split}_reads.json.gz',
                               set(part[KEY].itertuples(index=False, name=None)))
        status_text = 'OK' if part.label.nunique() == 2 else 'EMPTY' if part.empty else 'ONE_CLASS'
        if status_text != 'OK':
            warnings.warn(f'{name}/{split}: {status_text}')
        summaries.append(dict(dataset_id=name, split=split,
            manifest_sites=int(selected.split.eq(split).sum()), cleaned_sites=len(part),
            positive_sites=int(part.label.sum()), negative_sites=int(part.label.eq(0).sum()),
            groups=int(part.group_id.nunique()), status=status_text, output_file=str(path.resolve())))
    if coverage.status.ne('retained').any() or audited['_merge'].eq('left_only').any():
        warnings.warn(f'{name}: some sites were excluded or unmatched; inspect audit files')
    metadata = dict(dataset_id=name, config=config, manifest_path=str(manifest_path),
        manifest_sha256=MANIFEST_SHA256, source_label_checks=source_label_checks, source_path=str(source),
        source_bytes=source.stat().st_size, source_mtime_ns=source.stat().st_mtime_ns,
        max_lines=MAX_LINES, min_valid_reads=MIN_VALID_READS, require_drach=REQUIRE_DRACH,
        export_clean_reads=EXPORT_CLEAN_READS, parse_counts=parse_counts,
        read_count_mismatches=int(mismatch.sum()), removed_reads=int(sites.n_reads_removed.sum()),
        feature_columns=MODEL_FEATURES, summaries=summaries, python=sys.version,
        packages={p: safe_version(p) for p in ['numpy', 'pandas', 'matplotlib']},
        policy='Manifest binary labels unchanged; train+validation -> train; test -> test. No scaling/imputation/resampling.',
        partial_run=MAX_LINES is not None)
    (run_dir / 'run_manifest.json').write_text(json.dumps(metadata, indent=2, default=str))
    return summaries


## 5. Run all datasets and inspect the saved summary
Processing is sequential to limit memory usage. `manifest_sites` are counts before cleaning; `cleaned_sites` are counts actually exported. A run failure may leave completed earlier dataset folders; do not treat a partial run as complete.

In [ ]:
summaries = []
for config in DATASETS:
    summaries.extend(process_dataset(config))
    gc.collect()
run_summary = pd.DataFrame(summaries)
run_summary.to_csv(RUN_ROOT / 'run_summary.csv', index=False)
display(run_summary)
print('Complete. Use the six output_file paths shown above.')


## 6. Training and evaluation handoff
- Use each dataset's `*_clean_train.csv.gz` for training, cross-validation and tuning. `original_split` retains `train` or `validation`; `split` is now `train`.
- Use each `*_clean_test.csv.gz` only for final evaluation. Do not re-split it or use it to tune weights, thresholds, epochs or ensemble choices.
- Load predictors explicitly from `feature_columns.json`. Never use labels, modification fractions, identifiers, grouping columns or split metadata as predictors. Encode `central_5mer` in the model pipeline.
- Use the retained `group_id` when forming CV folds, keeping the same group together. Fix and share fold assignments across team models when comparing or creating out-of-fold ensemble predictions. Check both classes exist in every fold. Dataset2's grouping remains an assumption.
- Fit scalers, imputers, feature selectors and any resampling **within each training fold**. Compute data-derived class weights from that training fold. Final fitting uses the full train file after selecting settings from CV.
- CSVs retain 45 summaries over reads (mean, population SD, median, q25, q75 across nine measurements), valid read count and central 5-mer. Check your model notebook uses this same feature layout before training. A read-level model needs the optional per-split JSON exports instead.
- The six files remain separate. If a later model pools datasets, keep the shared group IDs intact and recheck biological overlap; do not prefix a shared gene group with dataset name.
- Review missing/unmatched/quarantined sites and report evaluation coverage. The prediction submission still needs original JSON site keys; this cleaning notebook alone does not supply fallback predictions for rejected sites.

Example (adapt the path):
```python
folder = RUN_ROOT / 'dataset0'
train = pd.read_csv(folder / 'dataset0_clean_train.csv.gz')
test = pd.read_csv(folder / 'dataset0_clean_test.csv.gz')
features = json.loads((folder / 'feature_columns.json').read_text())
X_train = train[features]
y_train = train['label']
groups = train['group_id']
```

## Validation and limitations
The actual supplied manifest was inspected. Notebook schema and Python syntax are checked, and the workflow is tested with small synthetic JSON fixtures to test split routing, feature preservation, invalid-read cleaning, unmatched sites, QC exclusions, and optional read exports. Synthetic fixtures are not biological results. Full cleaning cannot be executed here because the three raw JSON datasets were not supplied. To validate on your data, edit section 1 and run all cells; inspect `run_summary.csv` and audit files before using outputs. No model training occurs here.

This notebook updates the earlier cleaning workflow; retain acknowledgement of AI assistance and your team's checks as required by the handout.